In [0]:
incremental_path = "/Volumes/final_project/source/raw_data/loan_repayment_incremental_25.csv"


In [0]:
incremental_source_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(incremental_path)
)

display(incremental_source_df)

In [0]:
incremental_source_df.printSchema()

In [0]:
print("Columns:", incremental_source_df.columns)

In [0]:
print("Incremental Source Count:", incremental_source_df.count())

In [0]:
from pyspark.sql.functions import *



In [0]:
incremental_df = (
    incremental_source_df
    .withColumn("source_file", lit("loan_repayment_incremental_25.csv"))
    .withColumn("batch_id", lit("BATCH_002"))
    .withColumn("ingestion_timestamp", current_timestamp())
)



In [0]:
display(incremental_df)

In [0]:
incremental_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("final_project.incremental.loan_repayment_incremental_25")

print("Incremental Bronze table created successfully")

In [0]:
from pyspark.sql.window import Window

# Check duplicate payment IDs in incremental data
incremental_window = Window.partitionBy("payment_id")

incremental_validated_df = (
    incremental_df
    .withColumn(
        "duplicate_count",
        count("*").over(incremental_window)
    )
    .withColumn(
        "rejection_reason",
        concat_ws(
            ", ",
            when(
                col("payment_id").isNull() | (trim(col("payment_id")) == ""),
                lit("Missing payment_id")
            ),
            when(
                col("loan_id").isNull() | (trim(col("loan_id")) == ""),
                lit("Missing loan_id")
            ),
            when(
                col("customer_id").isNull() | (trim(col("customer_id")) == ""),
                lit("Missing customer_id")
            ),
            when(
                col("principal_amount").isNull() | (col("principal_amount") < 0),
                lit("Invalid principal_amount")
            ),
            when(
                col("interest_amount").isNull() | (col("interest_amount") < 0),
                lit("Invalid interest_amount")
            ),
            when(
                col("penalty_amount").isNull() | (col("penalty_amount") < 0),
                lit("Invalid penalty_amount")
            ),
            when(
                (coalesce(col("principal_amount"), lit(0)) <= 0) &
                (coalesce(col("interest_amount"), lit(0)) <= 0) &
                (coalesce(col("penalty_amount"), lit(0)) <= 0),
                lit("All amount components are zero")
            ),
            when(
                to_date(col("payment_date")).isNull(),
                lit("Invalid payment_date")
            ),
            when(
                col("loan_type").isNull() |
                (~upper(trim(col("loan_type"))).isin("HOME", "VEHICLE", "PERSONAL")),
                lit("Invalid loan_type")
            ),
            when(
                col("payment_status").isNull() |
                (~upper(trim(col("payment_status"))).isin("PENDING", "PAID", "FAILED")),
                lit("Invalid payment_status")
            ),
            when(
                col("duplicate_count") > 1,
                lit("Duplicate payment_id")
            )
        )
    )
)

display(incremental_validated_df)

In [0]:
# Separate incremental records

incremental_valid_df = incremental_validated_df.filter(
    col("rejection_reason") == ""
)

incremental_rejected_df = incremental_validated_df.filter(
    col("rejection_reason") != ""
)

print("Incremental Source Count:", incremental_source_df.count())
print("Incremental Valid Count:", incremental_valid_df.count())
print("Incremental Rejected Count:", incremental_rejected_df.count())

display(incremental_rejected_df)

In [0]:
existing_df = spark.table(
    "final_project.bronze.loan_repayment"
)

update_df = (
    incremental_df.join(
        existing_df.select("payment_id"),
        "payment_id",
        "left_semi"
    )
)

insert_df = (
    incremental_df.join(
        existing_df.select("payment_id"),
        "payment_id",
        "left_anti"
    )
)

print("Incoming :", incremental_df.count())
print("Updates  :", update_df.count())
print("Inserts  :", insert_df.count())

In [0]:
display(update_df)

In [0]:
display(insert_df)

In [0]:
incremental_silver_df = (
    incremental_valid_df
    .withColumn("payment_id", trim(col("payment_id")))
    .withColumn("loan_id", trim(col("loan_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("loan_type", upper(trim(col("loan_type"))))
    .withColumn("branch_city", upper(trim(col("branch_city"))))
    .withColumn("payment_status", upper(trim(col("payment_status"))))
    .withColumn("payment_date", to_date(col("payment_date")))
    .withColumn("principal_amount", col("principal_amount").cast("double"))
    .withColumn("interest_amount", col("interest_amount").cast("double"))
    .withColumn("penalty_amount", col("penalty_amount").cast("double"))
    .withColumn(
        "total_payment",
        col("principal_amount") +
        col("interest_amount") +
        col("penalty_amount")
    )
    .drop("rejection_reason", "duplicate_count")
    .dropDuplicates(["payment_id"])
)

display(incremental_silver_df)


In [0]:
# Read existing Silver table
existing_df = spark.table("final_project.bronze.loan_repayment")

# Identify existing payment IDs (update candidates)
update_df = incremental_silver_df.join(
    existing_df.select("payment_id"),
    on="payment_id",
    how="left_semi"
)

# Identify new payment IDs (insert candidates)
insert_df = incremental_silver_df.join(
   existing_df.select("payment_id"),
    on="payment_id",
    how="left_anti"
) 



In [0]:
display(update_df)


In [0]:
display(insert_df)

In [0]:
# Display counts
print("Silver Count Before MERGE:", existing_df.count())
print("Update Candidates:", update_df.count())
print("Insert Candidates:", insert_df.count())

print("Existing Payment Records:")
display(update_df)

print("New Payment Records:")
display(insert_df)

In [0]:
from delta.tables import DeltaTable

# Load existing Silver Delta table
silver_delta = DeltaTable.forName(
    spark,
    "final_project.silver.loan_repayment"
)

# Perform MERGE using payment_id
(
    silver_delta.alias("target")
    .merge(
        incremental_silver_df.alias("source"),
        "target.payment_id = source.payment_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Incremental MERGE completed successfully")

In [0]:
# Read updated Silver table
silver_after_merge_df = spark.table("final_project.silver.loan_repayment")

# Check final count
print("Silver Count Before MERGE:", existing_df.count())
print("Update Candidates:", update_df.count())
print("Insert Candidates:", insert_df.count())
print("Silver Count After MERGE:", silver_after_merge_df.count())
# Expected count calculation
expected_count = existing_df.count()- update_df.count()

print("Expected Silver Count:", expected_count)

if silver_after_merge_df.count() == expected_count:
    print("MERGE Validation: PASS")
else:
    print("MERGE Validation: FAIL")

In [0]:
# Verify newly inserted payment PAY0101

display(
    silver_after_merge_df
    .filter(col("payment_id") == "PAY0101")
)

In [0]:
# Find duplicate payment IDs

duplicate_check_df = (
    silver_after_merge_df
    .groupBy("payment_id")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate Payment IDs:", duplicate_check_df.count())

display(duplicate_check_df)

if duplicate_check_df.count() == 0:
    print("Duplicate Check: PASS")
else:
    print("Duplicate Check: FAIL")

In [0]:
# Refresh Gold summary from updated Silver data

gold_after_merge_df = (
    silver_after_merge_df
    .groupBy("loan_type")
    .agg(
        count("payment_id").alias("payment_count"),
        sum("principal_amount").alias("total_principal"),
        sum("interest_amount").alias("total_interest"),
        sum("total_payment").alias("total_collections")
    )
)

display(gold_after_merge_df)

In [0]:
gold_after_merge_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("final_project.gold.repayment_summary_by_loan_type")

print("Gold table refreshed successfully")


In [0]:
# BATCH_001 counts from the initial Bronze tables
source_count = spark.table("final_project.bronze.loan_repayment").count()
valid_count = spark.table("final_project.bronze.loan_repayment_valid").count()
rejected_count = spark.table("final_project.bronze.loan_repayment_rejected").count()
rejected_bronze_df = spark.table("final_project.bronze.loan_repayment_rejected")

print("INITIAL BATCH - BATCH_001")
print("--------------------------")
print("Source Count:", source_count)
print("Valid Count:", valid_count)
print("Rejected Count:", rejected_count)

print(
    "Reconciliation:",
    "PASS" if source_count == valid_count + rejected_count else "FAIL"
)

display(
    rejected_bronze_df.select(
        "payment_id",
        "loan_id",
        "customer_id",
        "rejection_reason"
    )
)

In [0]:
existing_df = spark.table("final_project.silver.loan_repayment")

print("INCREMENTAL BATCH - BATCH_002")
print("------------------------------")

print("Incremental Source Count:", incremental_source_df.count())
print("Incremental Valid Count:", incremental_valid_df.count())
print("Incremental Rejected Count:", incremental_rejected_df.count())



print("Silver Count After MERGE:", silver_after_merge_df.count())


expected_count = existing_df.count()

print("Expected Count:", expected_count)

print(
    "MERGE Status:",
    "PASS" if silver_after_merge_df.count() == expected_count else "FAIL"
)

In [0]:
# Verify final Silver and Gold tables

print("FINAL SILVER TABLE")
display(spark.table("final_project.silver.loan_repayment"))

print("FINAL GOLD TABLE")
display(
    spark.table("final_project.gold.repayment_summary_by_loan_type")
    .orderBy("loan_type")
)

# Check duplicate payment IDs
duplicate_count = (
    spark.table("final_project.silver.loan_repayment")
    .groupBy("payment_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate Payment IDs:", duplicate_count)
print("Duplicate Check:", "PASS" if duplicate_count == 0 else "FAIL")

In [0]:
# Display final Gold data for visualization

final_gold_df = spark.table(
    "final_project.gold.repayment_summary_by_loan_type"
)

display(final_gold_df)

In [0]:
display(
    final_gold_df.select(
        "loan_type",
        "payment_count"
    )
)

In [0]:
display(
    final_gold_df.select(
        "loan_type",
        "total_principal"
    )
)

In [0]:
display(
    final_gold_df.select(
        "loan_type",
        "total_interest"
    )
)